In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("amazon-reviews-cleaning")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

In [3]:
csv_path = "amazon_reviews.csv"

df_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("inferSchema", "false")
    .load(csv_path)
)

In [4]:
from pyspark.sql import functions as F

critical_cols = ["review_id", "product_id", "star_rating", "review_date"]


df_clean_1 = df_raw.na.drop(subset=critical_cols) 

print(df_clean_1.count())

396000


In [5]:
df_clean_2 = df_clean_1.withColumn(
    "review_date",
    F.to_date(F.col("review_date"), "yyyy-MM-dd")
)

df_clean_2.select("review_date").printSchema()
df_clean_2.select("review_date").show(5)

root
 |-- review_date: date (nullable = true)

+-----------+
|review_date|
+-----------+
| 2005-10-14|
| 2005-10-14|
| 2005-10-14|
| 2005-10-14|
| 2005-10-14|
+-----------+
only showing top 5 rows


In [6]:
df_verified = df_clean_2.filter(F.col("verified_purchase") == "1")
print("verified:", df_verified.count())

verified: 46823


In [ ]:
df = spark.read.csv(csv_path)
df.show(10)

